In [1]:
```python
# ================================================================
# RARE EVENTS ANALYSIS USING NLP + K-MEANS CLUSTERING
# ================================================================
#
# Dataset: rare_events.csv
#
# Workflow:
# 1. Load dataset
# 2. Explore dataset
# 3. Handle missing values and duplicates
# 4. Automatically identify text column
# 5. NLP text preprocessing
# 6. TF-IDF feature extraction
# 7. Find optimal K using Elbow Method and Silhouette Score
# 8. K-Means clustering
# 9. Evaluate clustering
# 10. Extract important words from clusters
# 11. Visualize clusters using PCA
# 12. Calculate cluster-based rarity score
# 13. Display rare events
# 14. Save results
# ================================================================


# ================================================================
# 1. INSTALL / IMPORT LIBRARIES
# ================================================================

# If required, uncomment and run:
# !pip install pandas numpy matplotlib scikit-learn nltk seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import re
import warnings

warnings.filterwarnings("ignore")

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

# Download NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("All libraries imported successfully.")


# ================================================================
# 2. LOAD DATASET
# ================================================================

FILE_NAME = "rare_events.csv"

try:
    df = pd.read_csv(FILE_NAME)
    print(f"\nDataset loaded successfully: {FILE_NAME}")

except FileNotFoundError:
    print(f"\nERROR: {FILE_NAME} was not found.")
    print("Make sure rare_events.csv is in the same folder as this notebook.")
    raise


# ================================================================
# 3. BASIC DATASET INFORMATION
# ================================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nNumber of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
for column in df.columns:
    print("-", column)

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# ================================================================
# 4. REMOVE DUPLICATES
# ================================================================

before_duplicates = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after_duplicates = len(df)

print(
    f"\nRemoved {before_duplicates - after_duplicates} duplicate rows."
)
print("Dataset shape:", df.shape)


# ================================================================
# 5. IDENTIFY TEXT COLUMN
# ================================================================

print("\n" + "=" * 70)
print("TEXT COLUMN DETECTION")
print("=" * 70)

# Common names for text columns
text_keywords = [
    "text",
    "description",
    "event",
    "event_description",
    "message",
    "comment",
    "content",
    "title",
    "summary",
    "details",
    "incident",
    "report"
]

# Find likely text columns
candidate_columns = []

for column in df.columns:

    column_lower = column.lower()

    if any(keyword in column_lower for keyword in text_keywords):

        if df[column].dtype == "object":
            candidate_columns.append(column)


# If no obvious column was found,
# find object/string columns automatically
if len(candidate_columns) > 0:

    TEXT_COLUMN = candidate_columns[0]

else:

    object_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()

    if len(object_columns) == 0:

        raise ValueError(
            "No text column was found. "
            "Your dataset must contain a text/string column."
        )

    # Choose the column with the largest average text length
    TEXT_COLUMN = max(
        object_columns,
        key=lambda col:
        df[col].fillna("").astype(str).str.len().mean()
    )


print("Selected text column:", TEXT_COLUMN)


# ================================================================
# 6. HANDLE MISSING TEXT
# ================================================================

df[TEXT_COLUMN] = (
    df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
)

print("\nMissing text values handled.")


# ================================================================
# 7. NLP PREPROCESSING
# ================================================================

print("\n" + "=" * 70)
print("NLP TEXT PREPROCESSING")
print("=" * 70)

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()


def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Keep only letters and spaces
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    # Remove very short words
    words = [
        word
        for word in words
        if len(word) > 2
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)


df["clean_text"] = df[TEXT_COLUMN].apply(clean_text)


print("\nOriginal text:")
print(df[TEXT_COLUMN].iloc[0])

print("\nCleaned text:")
print(df["clean_text"].iloc[0])


# ================================================================
# 8. REMOVE EMPTY TEXT RECORDS
# ================================================================

before_empty = len(df)

df = df[
    df["clean_text"].str.strip() != ""
].copy()

df = df.reset_index(drop=True)

after_empty = len(df)

print(
    f"\nRemoved {before_empty - after_empty} rows "
    "with empty text."
)

print("Remaining records:", len(df))


# ================================================================
# 9. TF-IDF FEATURE EXTRACTION
# ================================================================

print("\n" + "=" * 70)
print("TF-IDF FEATURE EXTRACTION")
print("=" * 70)


# TF-IDF converts text into numerical vectors
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)


try:

    X = tfidf_vectorizer.fit_transform(
        df["clean_text"]
    )

except ValueError:

    # Fallback for very small datasets
    print(
        "\nSmall dataset detected. "
        "Using a simpler TF-IDF configuration."
    )

    tfidf_vectorizer = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2)
    )

    X = tfidf_vectorizer.fit_transform(
        df["clean_text"]
    )


print("TF-IDF matrix shape:", X.shape)
print("Number of documents:", X.shape[0])
print("Number of features:", X.shape[1])


# ================================================================
# 10. FIND OPTIMAL NUMBER OF CLUSTERS
# ================================================================

print("\n" + "=" * 70)
print("FINDING OPTIMAL NUMBER OF CLUSTERS")
print("=" * 70)


# Do not test more clusters than the number of records
maximum_k = min(10, len(df) - 1)

if maximum_k < 2:

    raise ValueError(
        "At least 3 valid text records are required "
        "for K-Means clustering."
    )


k_values = list(range(2, maximum_k + 1))

inertia_values = []
silhouette_values = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    inertia_values.append(
        model.inertia_
    )

    silhouette_values.append(
        silhouette_score(X, labels)
    )


# Print scores
print("\nK     Inertia        Silhouette")

for k, inertia, silhouette in zip(
    k_values,
    inertia_values,
    silhouette_values
):

    print(
        f"{k:<5} "
        f"{inertia:<14.4f} "
        f"{silhouette:.4f}"
    )


# ================================================================
# 11. ELBOW METHOD GRAPH
# ================================================================

plt.figure(figsize=(9, 5))

plt.plot(
    k_values,
    inertia_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")

plt.xticks(k_values)
plt.grid(True)

plt.show()


# ================================================================
# 12. SILHOUETTE SCORE GRAPH
# ================================================================

plt.figure(figsize=(9, 5))

plt.plot(
    k_values,
    silhouette_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for K-Means")

plt.xticks(k_values)
plt.grid(True)

plt.show()


# ================================================================
# 13. SELECT BEST K
# ================================================================

best_index = np.argmax(
    silhouette_values
)

BEST_K = k_values[best_index]

print(
    "\nBest number of clusters according "
    f"to silhouette score: K = {BEST_K}"
)


# ================================================================
# 14. TRAIN FINAL K-MEANS MODEL
# ================================================================

print("\n" + "=" * 70)
print("K-MEANS CLUSTERING")
print("=" * 70)


kmeans = KMeans(
    n_clusters=BEST_K,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X)

df["cluster"] = cluster_labels

print("K-Means clustering completed successfully.")


# ================================================================
# 15. CLUSTER COUNTS
# ================================================================

print("\nRecords in each cluster:")

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

display(
    cluster_counts.to_frame(
        name="Number_of_Records"
    )
)


# ================================================================
# 16. CLUSTER PERCENTAGES
# ================================================================

cluster_percentages = (
    cluster_counts / len(df) * 100
).round(2)

cluster_summary = pd.DataFrame({
    "Number_of_Records": cluster_counts,
    "Percentage": cluster_percentages
})

print("\nCluster summary:")
display(cluster_summary)


# ================================================================
# 17. CLUSTERING EVALUATION
# ================================================================

print("\n" + "=" * 70)
print("CLUSTERING EVALUATION")
print("=" * 70)


silhouette = silhouette_score(
    X,
    cluster_labels
)

calinski = calinski_harabasz_score(
    X.toarray(),
    cluster_labels
)

davies = davies_bouldin_score(
    X.toarray(),
    cluster_labels
)


print(
    f"Silhouette Score       : {silhouette:.4f}"
)

print(
    f"Calinski-Harabasz     : {calinski:.4f}"
)

print(
    f"Davies-Bouldin Score  : {davies:.4f}"
)


# ================================================================
# 18. TOP WORDS FOR EACH CLUSTER
# ================================================================

print("\n" + "=" * 70)
print("TOP WORDS IN EACH CLUSTER")
print("=" * 70)


feature_names = (
    tfidf_vectorizer
    .get_feature_names_out()
)

cluster_centers = (
    kmeans.cluster_centers_
)

for cluster_number in range(BEST_K):

    # Sort words according to cluster importance
    top_indices = cluster_centers[
        cluster_number
    ].argsort()[::-1][:15]

    top_words = [
        feature_names[index]
        for index in top_indices
    ]

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )


# ================================================================
# 19. DISPLAY SAMPLE EVENTS FROM EACH CLUSTER
# ================================================================

print("\n" + "=" * 70)
print("SAMPLE EVENTS FROM EACH CLUSTER")
print("=" * 70)


for cluster_number in range(BEST_K):

    print(
        f"\n{'=' * 20} "
        f"CLUSTER {cluster_number} "
        f"{'=' * 20}"
    )

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    display(
        cluster_data[
            [
                TEXT_COLUMN,
                "cluster"
            ]
        ].head(10)
    )


# ================================================================
# 20. PCA VISUALIZATION
# ================================================================

print("\n" + "=" * 70)
print("PCA CLUSTER VISUALIZATION")
print("=" * 70)


# PCA requires dense matrix
X_dense = X.toarray()

# For very large datasets, sample the data
MAX_PCA_POINTS = 5000

if len(df) > MAX_PCA_POINTS:

    rng = np.random.RandomState(42)

    sample_indices = rng.choice(
        len(df),
        MAX_PCA_POINTS,
        replace=False
    )

    X_pca_input = X_dense[
        sample_indices
    ]

    labels_pca = df["cluster"].iloc[
        sample_indices
    ]

else:

    X_pca_input = X_dense

    labels_pca = df["cluster"]


pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(
    X_pca_input
)


plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=labels_pca,
    alpha=0.7
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.title(
    "K-Means Clustering of Rare Events"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(True)

plt.show()


print(
    "Explained variance:",
    pca.explained_variance_ratio_
)


# ================================================================
# 21. CLUSTER SIZE VISUALIZATION
# ================================================================

plt.figure(figsize=(9, 5))

cluster_counts.plot(
    kind="bar"
)

plt.xlabel("Cluster")
plt.ylabel("Number of Events")

plt.title(
    "Number of Events in Each Cluster"
)

plt.xticks(rotation=0)
plt.grid(axis="y")

plt.show()


# ================================================================
# 22. CALCULATE RARITY SCORE
# ================================================================
#
# A small cluster is considered more unusual at the
# cluster level than a large cluster.
#
# This is NOT a statistical anomaly score.
# It is a simple cluster-size-based rarity score.
# ================================================================

cluster_size = df["cluster"].map(
    cluster_counts
)

df["cluster_size"] = cluster_size

# Basic rarity
df["rarity_score_raw"] = (
    1 / df["cluster_size"]
)


# Normalize to 0-1
minimum_score = (
    df["rarity_score_raw"].min()
)

maximum_score = (
    df["rarity_score_raw"].max()
)


if maximum_score > minimum_score:

    df["rarity_score"] = (
        (df["rarity_score_raw"] - minimum_score)
        /
        (maximum_score - minimum_score)
    )

else:

    df["rarity_score"] = 0.0


# ================================================================
# 23. RANK RARE EVENTS
# ================================================================

df["rarity_rank"] = (
    df["rarity_score"]
    .rank(
        ascending=False,
        method="dense"
    )
)


print("\n" + "=" * 70)
print("MOST RARE EVENTS BASED ON CLUSTER SIZE")
print("=" * 70)


rare_events = df.sort_values(
    "rarity_score",
    ascending=False
)


display(
    rare_events[
        [
            TEXT_COLUMN,
            "cluster",
            "cluster_size",
            "rarity_score",
            "rarity_rank"
        ]
    ].head(20)
)


# ================================================================
# 24. SHOW SMALLEST CLUSTERS
# ================================================================

smallest_clusters = (
    cluster_counts
    .sort_values()
)

print("\nSmallest clusters:")

display(
    smallest_clusters.to_frame(
        name="Number_of_Records"
    )
)


# ================================================================
# 25. SAVE CLUSTERED DATASET
# ================================================================

OUTPUT_FILE = (
    "rare_events_clustered.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nClustered dataset saved as: "
    f"{OUTPUT_FILE}"
)


# ================================================================
# 26. SAVE ONLY RARE EVENTS
# ================================================================

RARE_OUTPUT_FILE = (
    "rare_events_identified.csv"
)


# Define rare as events belonging to
# clusters smaller than the median cluster size
median_cluster_size = (
    cluster_counts.median()
)

rare_df = df[
    df["cluster_size"] <= median_cluster_size
].copy()


rare_df = rare_df.sort_values(
    "rarity_score",
    ascending=False
)


rare_df.to_csv(
    RARE_OUTPUT_FILE,
    index=False
)


print(
    f"Rare-event dataset saved as: "
    f"{RARE_OUTPUT_FILE}"
)

print(
    "Number of identified rare events:",
    len(rare_df)
)


# ================================================================
# 27. FINAL RESULTS
# ================================================================

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    "Original dataset size:",
    before_duplicates
)

print(
    "Final dataset size:",
    len(df)
)

print(
    "Text column:",
    TEXT_COLUMN
)

print(
    "TF-IDF features:",
    X.shape[1]
)

print(
    "Number of clusters:",
    BEST_K
)

print(
    f"Silhouette Score: {silhouette:.4f}"
)

print(
    "Clustered output:",
    OUTPUT_FILE
)

print(
    "Rare events output:",
    RARE_OUTPUT_FILE
)


# ================================================================
# 28. FINAL DATASET PREVIEW
# ================================================================

print("\nFinal dataset preview:")

display(
    df.head(20)
)


# ================================================================
# END
# ================================================================

print("\nAnalysis completed successfully.")
```

### Expected output files

After running the notebook, you will get:

* **`rare_events_clustered.csv`** — all events with their K-Means cluster and rarity score.
* **`rare_events_identified.csv`** — events belonging to the smaller clusters.
* Graphs for the **Elbow Method**, **Silhouette Score**, **PCA clusters**, and **cluster sizes**.
* A list of the **top NLP keywords** describing each cluster.
* **Silhouette, Calinski-Harabasz, and Davies-Bouldin** clustering metrics.

**Important:** this assumes `rare_events.csv` contains at least one text/string column. If you upload `rare_events.csv` here, I can adapt the code to its **actual column names and data structure** rather than relying on automatic column detection.


SyntaxError: invalid character '—' (U+2014) (922364822.py, line 994)